In [22]:
# 여기부터 run

from dotenv import load_dotenv
import pymysql
import os
import pandas as pd

In [23]:
load_dotenv()

True

In [24]:
# class 선언 
class MyDB:
    # 생성자 함수 (서버의 정보를 받아오기 위해 사용)
    def __init__(self, host, port, user, password, db):
        # 서버의 정보를 인자로 받아와서 객체 내부에서 독립적인 변수에 저장
        self.host = host
        self.port = port
        self.user = user
        self.password = password
        self.db = db

    # 데이터베이스에 변화를 주는 함수 
    def commit(self):
        try:
            # DataBase에 데이터를 확정 (동기화)
            self.db_server.commit()
            # 서버와의 연결을 종료 ( 중요한 부분 )
            self.db_server.close()
            # close() 함수를 사용하더라도 self.db_server의 변수는 사라지지 않는다. 
            # 변수 자체를 제거 
            del self.db_server
        except:
            # 문제가 발생하는 이유는? -> 서버와의 연결이 되지 않은 경우 (self.db_server라는 변수에 데이터가 없거나 아예 존재하지 않는 경우)
            print( "데이터베이스 서버와의 연결이 되어있지 않습니다. sql_query() 함수를 호출하여 서버와의 연결을 해주세요" )
    
    def sql_query(self, query, *datas):
        # query : sql query문이 입력이되는 매개변수 
        # datas : query문에서 사용이 될 데이터의 목록

        # 문제점 : 서버의 재접속으로 commit 전의 데이터가 날아감. 
        # 이미 접속중인 경우 재접속 금지 (2026.04.20 update)
        # 해결 방법 self.db_server에 데이터가 존재한다면? -> 서버의 접속중이다. 
        try:
            self.db_server
            # 변수가 존재하지 않으면 NameError 발생 
            print('접속된 서버가 존재함')
        except:
            # 예외 상황이 발생하면 서버와 연결(self.db_server 라는 변수가 없을 시 실행)
            # DB_server와의 연결 
            self.db_server = pymysql.connect(
                host = self.host, 
                port = self.port, 
                user = self.user, 
                password = self.password, 
                db = self.db
            )
        # cursor 생성 
        cursor = self.db_server.cursor(pymysql.cursors.DictCursor)

        # self.변수명 // 변수명의 차이는?
            # self.변수 -> 독립적으로 객체 안에 저장이 되는 변수 ( 함수 호출 후에도 데이터가 존재 )
            # 변수 -> 함수 호출시 생성이 되고 함수가 종료가 되면 휘발성으로 사라짐
        try:
            # CUD의 경우에는 execute() 쿼리문을 커서에 질의 보낸다. R (select도 여기까지는 공통의 작업)
            # execute( query, () ) -> 호출 가능 
            # execute( query, (1,2,3) ) -> 호출 가능 
            cursor.execute(query, datas)
            # query가 select문이라면? 
            # select * from table // SELECT * FROM table, """   select * from table   """ -> 두가지의 경우 모두 참 
            # 좌측 공백을 제거 , 소문자를 통일 , 시작값이 select와 같은가 startwith()
            if query.lstrip().lower().startswith('select'):
                # 결과값을 받아온다. 
                result = cursor.fetchall()
            else:
                result = "Query OK!"
            return result
        except Exception as e:
            print('query문 execute중 에러')
            print(e)

In [25]:
# class 생성

db1= MyDB(
    host = os.getenv('host'),
    port = int(os.getenv('port')),
    user = os.getenv('user'),
    password = os.getenv('pwd'),
    db = os.getenv('db_name')
)

In [29]:
db1.sql_query('select * from `emp`')

접속된 서버가 존재함


[{'EMPNO': 7369,
  'ENAME': 'SMITH',
  'JOB': 'CLERK',
  'MGR': 7902.0,
  'HIREDATE': '1980-12-17',
  'SAL': 800.0,
  'COMM': 0.0,
  'DEPTNO': 20.0},
 {'EMPNO': 7499,
  'ENAME': 'ALLEN',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-02-20',
  'SAL': 1600.0,
  'COMM': 300.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7521,
  'ENAME': 'WARD',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-02-22',
  'SAL': 1250.0,
  'COMM': 500.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7566,
  'ENAME': 'JONES',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
  'HIREDATE': '1981-04-02',
  'SAL': 2975.0,
  'COMM': 0.0,
  'DEPTNO': 20.0},
 {'EMPNO': 7654,
  'ENAME': 'MARTIN',
  'JOB': 'SALESMAN',
  'MGR': 7698.0,
  'HIREDATE': '1981-09-28',
  'SAL': 1250.0,
  'COMM': 1400.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7698,
  'ENAME': 'BLAKE',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
  'HIREDATE': '1981-05-01',
  'SAL': 2850.0,
  'COMM': 0.0,
  'DEPTNO': 30.0},
 {'EMPNO': 7782,
  'ENAME': 'CLARK',
  'JOB': 'MANAGER',
  'MGR': 7839.0,
 

In [31]:
# select 확인이 끝났으니 insert update delete 확인 
insert_query = """
    INSERT INTO `user_info`
    VALUES (%s, %s, %s, %s)
"""
select_query = """
    SELECT * FROM `user_info`
"""

data_list = ['test3', '0000', 'lee', 40]
db1.sql_query(insert_query, *data_list)

접속된 서버가 존재함


'Query OK!'

In [32]:
db1.sql_query(select_query)

접속된 서버가 존재함


[{'id': 'test', 'password': '1234', 'name': 'kim', 'age': 30},
 {'id': 'test3', 'password': '0000', 'name': 'lee', 'age': 40}]

In [33]:
delete_query = """
    DELETE FROM `user_info`
"""

data_list = ['test']

db1.sql_query(delete_query)

접속된 서버가 존재함


'Query OK!'